# SQLBuddy 04 - Live demo on GPU (backup for the presentation)
Runs the same Gradio app as `app/app.py` on a Kaggle T4 and prints a public `gradio.live` link.
The fine-tuned adapter comes from the `02_train_lora` kernel output (no Hugging Face token needed).
**Use:** open this notebook in the Kaggle editor (GPU T4, Internet on) -> *Run all* -> open the
`*.gradio.live` link printed below. The app stays up for `DEMO_MINUTES`.

In [ ]:
%pip uninstall -y -q torchao
%pip install -q transformers==5.22.1 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0 huggingface_hub==2.4.0 nltk==3.9.2 gradio==6.31.0

In [ ]:
# Locate our shared code (Kaggle dataset "sqlbuddy-src") and the Spider databases, then import.
import glob, os, shutil, sys, zipfile
work = "/kaggle/working/sqlbuddy"
# The package folder is the one containing run_eval.py + spider_eval/ (Kaggle may unpack it at the
# dataset root or keep a src/ folder); copy it to /kaggle/working/sqlbuddy/src.
hits = [h for h in glob.glob("/kaggle/input/**/run_eval.py", recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(h), "spider_eval"))]
if hits:
    shutil.copytree(os.path.dirname(hits[0]), work + "/src", dirs_exist_ok=True)
else:  # or it may still be a zip archive
    zips = glob.glob("/kaggle/input/**/src.zip", recursive=True)
    assert zips, "sqlbuddy-src dataset not attached"
    zipfile.ZipFile(zips[0]).extractall(work + "/src")
sys.path.insert(0, work)
print("src commit:", open(work + "/src/GIT_COMMIT").read().strip() if os.path.exists(work + "/src/GIT_COMMIT") else "?")

from src.data import find_spider_root
SPIDER_ROOT = find_spider_root("/kaggle/input")
print("Spider root:", SPIDER_ROOT)

import torch, transformers, peft, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| datasets", datasets.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

# demo app + sample database (shipped in the sqlbuddy-src dataset)
core = glob.glob("/kaggle/input/**/sqlbuddy_core.py", recursive=True)[0]
shutil.copytree(os.path.dirname(core), work + "/app", dirs_exist_ok=True)
os.makedirs(work + "/sample_db", exist_ok=True)
shutil.copy(glob.glob("/kaggle/input/**/university.db", recursive=True)[0], work + "/sample_db/")

In [ ]:
ADAPTER = os.path.dirname([p for p in glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
                           if "checkpoint" not in p][0])
os.environ["SQLBUDDY_ADAPTER"] = ADAPTER   # read by app/sqlbuddy_core.py at import time
sys.path.insert(0, work + "/app")
import app as sqlbuddy_app
sqlbuddy_app.MODELS.load()                 # load base + adapter on the GPU now, not on the first click
print("adapter loaded:", sqlbuddy_app.MODELS.has_adapter, sqlbuddy_app.MODELS.adapter_error or "")

In [ ]:
# Self-test: one example question through both models (same code path as the UI button)
q = "What is the average score in each course? Show the course title."
db = str(sqlbuddy_app.SAMPLE_DB)
for ft in (False, True):
    sql, df, status = sqlbuddy_app.answer_one(db, q, ft)
    print(status)
    print("  ", sql)
    print(df.head(3))
    print()

In [ ]:
import time
DEMO_MINUTES = 1
demo = sqlbuddy_app.build_ui()
demo.launch(share=True, prevent_thread_lock=True, css=sqlbuddy_app.CSS)
print("PUBLIC LINK:", demo.share_url)      # open this *.gradio.live URL in any browser
time.sleep(DEMO_MINUTES * 60)
demo.close()